In [2]:
import sqlite3
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
import pandas as pd
from imblearn.over_sampling import RandomOverSampler
import seaborn as sns
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.utils.validation import check_is_fitted
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree

ModuleNotFoundError: No module named 'pandas'

In [ ]:
df = pd.read_csv('/content/framingham.csv')

In [ ]:
df.head(10)

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df.tail()

In [ ]:
x = df.drop(columns = 'TenYearCHD')
y = df.TenYearCHD

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42, stratify = y)

x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size = 0.25, random_state = 42, stratify = y_train)

In [ ]:
"""
What is stratify: it keeps the same class proportion for very split. stratify parameter ensures that the split datasets maintain the exact same proportion of class labels as the original dataset
we need to hyperparameter tuning for making a good model.
train test split->
test is the final exam, -> 20%
train is the data model will train -> 64%
for hyperparameter tuning we will do validation on the 16%(25% from train data) of the data. -> 16%
we need to make sure that the model see the test data only once in the last.
"""

In [ ]:
model = make_pipeline(
    SimpleImputer(strategy = 'median'),
    DecisionTreeClassifier(
        max_depth = 25,
        random_state = 42
    )
)

model.fit(x_train, y_train)

In [ ]:
model.score(x_val, y_val)

In [ ]:
model.score(x_train, y_train)

In [ ]:
val_acc = []
train_acc =[]

for d in range(1, 26):
  model = make_pipeline(SimpleImputer(), DecisionTreeClassifier(max_depth = d, random_state = 42))
  model.fit(x_train, y_train)
  val_acc.append(model.score(x_val, y_val))
  train_acc.append(model.score(x_train, y_train))


plt.plot(range(1, 26), val_acc, label = "Validation Accuracy")
plt.plot(range(1, 26), train_acc, label = "Training Accuracy")
plt.xlabel("Depth")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy vs Tree Depth")
plt.grid()
plt.legend()

In [ ]:
over_sampler = RandomOverSampler(random_state = 42)
x_train_over, y_train_over = over_sampler.fit_resample(x_train, y_train)
print(x_train.shape)
print(x_train_over.shape)

### this used to address class imbalance, where one class (majority) heavily outnumbers
### often used in fraud detection and medical diagnosis.
### WE ONVER SAMPLE ONLY TRAINING.

In [ ]:
"""
Random Forest:
many decision trees,
each trained on random subset of data
final prediction = voting from all trees
"""

In [ ]:
model2 = make_pipeline(SimpleImputer(), RandomForestClassifier())
print(model2)

In [ ]:
cv_acc_scores = cross_val_score(model2, x_train_over, y_train_over, cv = 5, n_jobs = -1)
print(cv_acc_scores)
print("Mean CV accuracy", cv_acc_score.mean())

In [ ]:
param = {
    "simpleimputer_strategy" : ["mean", "median"],
    "randomforestclassifier__n_estimators" : [10, 100],
    "randomforestclassifier__max_depth" : [2, 10, 30]
}

In [ ]:
model_main = GridSearchCV(
    model2,
    param_grid = param,
    cv = 5,
    n_jobs = -1,
    verbose = 1,
    scoring = "roc_auc"
)

### we will use grid search cv (cross valuation) to calculate every possible combination

In [ ]:
model_main.fit(x_train_over, y_train_over)

In [ ]:
cv_results = pd.DataFr